# Geo-RAG Colab Server (Qwen2.5-VL-7B-AWQ)
Run this notebook on Google Colab with a T4 GPU.

In [ ]:
!pip install -q autoawq transformers accelerate fastapi uvicorn cloudflared pydantic qwen-vl-utils slowapi

# УБЕДИТЕСЬ, что файл app.py загружен в корневую папку Colab перед запуском следующих ячеек.

In [ ]:
import subprocess
import time
import re

# Скачиваем и запускаем cloudflared в фоне, направляя логи в файл
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Запуск в фоновом режиме через nohup
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8000 > cloudflared.log 2>&1 &

time.sleep(8)  # Даем время на установку туннеля

try:
    with open('cloudflared.log', 'r') as f:
        logs = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
        if match:
            print("\n" + "="*60)
            print(f"YOUR COLAB TUNNEL URL:\n{match.group(0)}")
            print("Copy this URL into your Streamlit UI or .env file.")
            print("="*60 + "\n")
        else:
            print("Tunnel URL not found yet. Check cloudflared.log manually.")
except FileNotFoundError:
    print("Log file not created. Cloudflared might have failed to start.")

In [ ]:
import os
# Установите ваш секретный ключ!
os.environ['COLAB_API_KEY'] = 'your_secure_api_key_here'

# Запускаем сервер uvicorn в основном потоке. Это решит проблемы с таймаутами потоков.
# Модель будет загружена синхронно (2-3 минуты), после чего сервер начнет принимать запросы.
!COLAB_API_KEY=$COLAB_API_KEY uvicorn app:app --host 0.0.0.0 --port 8000